# 02 — Stationarity Testing

**Goal:** Determine whether VIX levels require differencing before fitting ARIMA-family models.

We use **ADF** and **KPSS** together as a cross-check, because the two tests have opposite null hypotheses and opposite failure modes:

| Test | H₀ | Reject H₀ means... |
|------|----|--------------------|
| ADF  | Unit root (non-stationary) | Series is stationary |
| KPSS | Stationary | Series is non-stationary |

Agreement between them gives confidence. If they disagree, the series may be fractionally integrated or structurally broken — requiring further investigation.

In [ ]:
import sys, warnings
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

sys.path.insert(0, str(Path('..').resolve()))
warnings.filterwarnings('ignore')

from src.vix_forecasting.data.preprocessing import load_raw, build_series
from src.vix_forecasting.analysis.stationarity import run_adf, run_kpss, print_result

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')

vix  = build_series(load_raw(Path('../data/raw/vix_raw.csv')))
diff1 = vix.diff().dropna()
print('Ready.')

## Visual inspection

Before running formal tests, a visual comparison of the level vs. first-differenced series helps calibrate expectations.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)

axes[0].plot(vix.index, vix.values, color='#2B6CB0', linewidth=0.7)
axes[0].set_title('VIX — Level series')
axes[0].set_ylabel('VIX')

axes[1].plot(diff1.index, diff1.values, color='#2B9348', linewidth=0.7)
axes[1].axhline(0, color='gray', linewidth=0.8, linestyle='--')
axes[1].set_title('VIX — First difference')
axes[1].set_ylabel('ΔVIX')

fig.tight_layout()
fig.savefig('../figures/vix_level_vs_diff.png', dpi=150)
plt.show()

## Level series

In [ ]:
adf_level  = run_adf(vix,  regression='c')
kpss_level = run_kpss(vix, regression='c')

print('--- ADF ---')
print_result(adf_level)
print()
print('--- KPSS ---')
print_result(kpss_level)

**Interpretation — Level series:**

- **ADF**: statistic = −7.31, p ≈ 0.000. Strongly rejects the unit-root null → evidence for stationarity.
- **KPSS**: statistic = 0.348, p ≈ 0.10. Fails to reject the stationarity null at the 5% level (critical value at 5% is 0.463). The warning from statsmodels notes the actual p-value may be even higher. Evidence for stationarity.
- **Both tests agree: the VIX level series is stationary.**

This might seem surprising given the slow ACF decay observed in EDA, but it is consistent with the economics: VIX is a *fear index* anchored to near-term option prices. It cannot trend indefinitely — extreme readings are self-correcting because high implied volatility makes options expensive, suppressing demand. The series exhibits **high persistence** (AR coefficient close to 1) without a true unit root.

**Decision: model VIX in levels (d = 0). No differencing is needed.**

## First difference — confirmation check

In [ ]:
adf_diff  = run_adf(diff1,  regression='c')
kpss_diff = run_kpss(diff1, regression='c')

print('--- ADF on first difference ---')
print_result(adf_diff)
print()
print('--- KPSS on first difference ---')
print_result(kpss_diff)

**Interpretation — First difference:**

Both tests confirm stationarity with very high confidence (ADF statistic = −19.59, KPSS statistic = 0.008). This is expected: if the level is stationary, the first difference is trivially stationary too — it removes whatever persistence remained.

Differencing here would be unnecessary and harmful: it would destroy the mean-reverting signal we want to exploit, and introduce an MA(1) component into the error structure that inflates model complexity without adding predictive power.

## Summary

| Series | ADF conclusion | KPSS conclusion | Decision |
|--------|---------------|-----------------|----------|
| VIX level | Stationary (p ≈ 0.000) | Stationary (p ≈ 0.10) | **Use levels** |
| First difference | Stationary (p ≈ 0.000) | Stationary (p ≈ 0.10) | Over-differenced |

**ARIMA order implication:** d = 0. Combined with the PACF profile from EDA (sharp cutoff after lag 1–2), the working hypothesis entering Phase 5 is **ARIMA(1,0,0)** or **ARIMA(2,0,0)** — i.e., an AR(1) or AR(2) model on the raw VIX level. We will confirm this via information criteria (AIC/BIC) when fitting.